# Машину для нового семейства находит сама система

`research_m5/cells.py` — один обучатель для любого семейства задач. Семейство для него — только символы и ответы; что это за задачи, ему не говорят, и ни одной машины под семейство в коде нет.

**Что он ищет.** Машину из маленького полного базиса (Крон и Роудс, 1965): конечное управление (k состояний) и рядом до двух чисел — целое (счётчик, читается обрезанным до знака или до ±2) или круг по модулю m (дни недели, курс). Каждый символ двигает управление по таблице и прибавляет к числам смещение, которое может зависеть от состояния управления (зависимый каскад: курс и положение). Ответ — таблица от того, что видно в конце.

**Как.** Формы перебираются в порядке длины описания в битах; для каждой решатель SAT (кодировка Heule и Verwer, 2010; нумерация состояний обходом в ширину — Ульянцев и др., 2015; целые в порядковой кодировке — Tamura и др., 2009) ищет машину, которая отвечает на все учебные рассказы верно. Первая найденная остаётся. Потом обучатель **сам спрашивает** (Angluin, 1987): решатель ищет вторую машину не длиннее чем на 8 бит, которая тоже объясняет всё выученное, но на каком-то рассказе отвечает иначе, — и обучатель спрашивает ответ ровно на этот рассказ. Когда второй машины нет, он спрашивает ещё шесть длинных рассказов для проверки.

**Протокол.** Учение — 300 рассказов длиной 1–4 символа и не больше 40 спрошенных; проверка — 300 рассказов длиной 20, 100 и 500. Соперники — LSTM и трансформер на тех же 300 рассказах и на в 40 раз большем числе; засчитываются, только если на учебных длинах дают не меньше 95 %.

Первые ячейки видеокарты не требуют. Короткая проверка, что всё запускается: `%env MX_QUICK=1` до первой ячейки.

In [ ]:
import os, subprocess
ROOT = os.environ.get('MX_ROOT', '/content')
QUICK = bool(os.environ.get('MX_QUICK'))
if os.path.exists(ROOT + '/MX-Project'):
    subprocess.run(['git', '-C', ROOT + '/MX-Project', 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/kireikosasha/MX-Project', ROOT + '/MX-Project'], check=True)
M5 = ROOT + '/MX-Project/research_m5'
!pip -q install python-sat
G = ROOT + '/clutrr_graph'
os.makedirs(G, exist_ok=True)
RAW = 'https://raw.githubusercontent.com/azreasoners/LLM-ASP/main/clutrr/data/clutrr/data_089907f8'
for f in ['1.2%2C1.3_train.csv'] + [f'1.{k}_test.csv' for k in range(2, 11)]:
    out = G + '/' + f.replace('%2C', ',')
    if not os.path.exists(out):
        !wget -q -O "{out}" "{RAW}/{f}"
WORKERS = os.cpu_count()

## Десять семейств одним обучателем

Что вышло на процессоре (README, этап 47; ожидание для Colab — то же, всё точное): семь семейств с одной-двумя клетками — 100 % на длине 500 после 9–15 вопросов, за секунды; «черепаха» (курс из четырёх состояний и два целых числа, шаг зависит от курса) — 100 %, около 20 минут; одометр (день недели и счёт недель с переносом) — 100 %, но вопросов больше 40; навигация («вернулся ли домой») — не взята (96–99 % после 40 вопросов). «Черепаха» и одометр — самые долгие ячейки (до часа на процессоре).

In [ ]:
fams = 'parity lies weekday latest majority shells' if QUICK else 'parity lies weekday latest majority shells firstlast odometer navigate turtle'
!cd {M5} && python cells.py {fams} --workers {WORKERS} | grep -v "gave up"

## Тот же обучатель на опубликованном мериле: граф CLUTRR (здесь он пока проигрывает)

Цепочки родства — слова набора, ни одно не объяснено; учение на цепочках из 2–3 звеньев, проверка на 2–10. Первая строка — обучатель целиком: 100 цепочек и до 40 вопросов о других учебных; на процессоре он нашёл дешёвую машину, которая объясняет выученное, но не родство (24 %). Вторая — форма дана («состояния — это сами ответы», таблица отношений из базиса), все 383 разные цепочки, наименьшее число исключений, при котором такая машина есть (выбрано по учебным данным): на процессоре 75 % на всех тестах и 54 % на 10 звеньях. Рукописная таблица прежних этапов — 100 %; опубликовано (статья CTP): GAT 45 %, LSTM 75 %, CTP 90 %. Почему проигрыш: в наборе нет цепочек из одного звена, значение отдельного слова нигде не сказано, и спросить некого. Ожидание для Colab — те же числа (всё точное, от видеокарты не зависит).

In [ ]:
!cd {M5} && python cells.py --clutrr {G} --named
!cd {M5} && python cells.py --clutrr {G} --teach 100 --asks 40 --workers {WORKERS} | grep -v "gave up"

## Соперники на видеокарте

LSTM и трансформер (4 слоя, ширина 128; затем 6 слоёв, ширина 256) на тех же 300 рассказах и на 12 000 (в 40 раз больше), три зерна. Ожидание: на учебных длинах 95–100 %, на длине 100 и 500 — около угадывания у семейств, где ответ зависит от всего рассказа (на процессоре так и вышло). Опровержение: хотя бы одна сеть даёт не меньше 90 % на длине 100 на одном из семейств, кроме чётности, «последнего значения» и «первого = последнего», которые LSTM, бывает, берёт.

In [ ]:
fams = 'parity weekday' if QUICK else 'parity lies weekday latest majority shells firstlast odometer navigate turtle'
steps = 200 if QUICK else 3000
!cd {M5} && python cells_rivals.py {fams} --steps {steps} --seeds 3 --times 1 40 | grep "mean of"
!cd {M5} && python cells_rivals.py {fams} --steps {steps} --seeds 3 --times 40 --kinds transformer --width 256 --layers 6 | grep "mean of"